# NLP Mastery Journey — Module 6: Neural Networks for NLP (RNNs, LSTMs, and the Road to Transformers)

Module 5's models are fast and strong baselines, but they share one blind spot: `CountVectorizer`/`TfidfVectorizer` features are a **bag** — `"not good"` and `"good not"` produce identical features (n-grams patch this locally, but not for long-range order). This module builds the family of models that reads text **sequentially, in order**, and ends exactly at the doorstep of the architecture behind every modern LLM, including Claude: the **Transformer**.

### What this notebook teaches
| # | Topic | Why it matters |
|---|-------|------------------|
| 1 | Why sequence matters | The concrete gap classical ML leaves open |
| 2 | RNN fundamentals | The recurrence idea — a hidden state carried across a sentence |
| 3 | Building & training an RNN classifier | End-to-end PyTorch, on real (small) data |
| 4 | LSTM & GRU | The gating mechanism that fixes RNNs' vanishing-gradient problem |
| 5 | Bidirectional RNNs | Reading context from both directions at once |
| 6 | Sequence labeling (NER/POS-style) | The other major use of RNNs: tagging every token, not just the sentence |
| 7 | Attention | The idea that broke RNNs' long-range bottleneck — and directly seeded Transformers |
| 8 | Why Transformers replaced RNNs | The specific limitation (no parallelism) that made the switch inevitable |
| 9 | Production patterns | Padding, packing, batching, checkpointing — the engineering, not just the math |

### How to use this notebook
- This module needs **PyTorch** — install it once with the cell below. Everything is written to run **on CPU**, since our teaching dataset is tiny; a GPU only starts to matter once you scale up to real corpora.
- Every code cell is commented line-by-line, same as every module before this.
- **🔀 Alternatives** and **📋 Copy-paste template** callouts continue as before.
- We reuse the exact sentiment dataset from Module 5, so you can directly compare "classical ML accuracy" vs. "RNN accuracy" on identical data — that comparison is itself a valuable, realistic lesson (spoiler: on a dataset this small, classical ML often still wins — see Part 9).


## 0. Setup

In [ ]:
# %pip install torch --index-url https://download.pytorch.org/whl/cpu   # CPU-only build,
                                                                            # smaller download,
                                                                            # plenty for this notebook
# %pip install scikit-learn pandas numpy matplotlib

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("Setup note: uncomment the pip install lines above the first time you run this.")
print("PyTorch version:", torch.__version__ if hasattr(torch, "__version__") else "not installed yet")


## Part 1 — Why Sequence Order Needs Its Own Architecture

Take two sentences: `"the movie was not good, it was actually quite bad"` and imagine a version with `"not"` moved: `"the movie was good, it was not actually quite bad"`. A bag-of-words model sees **the exact same word counts** for wildly different meanings once negation gets separated from what it's negating. Bigrams help locally, but a negation word 5+ tokens away from what it modifies is invisible to any n-gram window you'd realistically use.

**The core idea of this module**: instead of throwing words into an unordered bag, process the sentence **one token at a time, in order**, carrying forward a running summary (a "hidden state") of everything read so far. That's a Recurrent Neural Network (RNN) in one sentence — everything else in this module is refining that one idea.


## Part 2 — Dataset & Vocabulary

Same synthetic sentiment dataset as Module 5 (hand-written + templated reviews, with a touch of label noise for realism) — so results here are directly comparable to your classical-ML numbers.


In [ ]:
hand_written_positive = [
    "this movie was absolutely fantastic and I loved every minute of it",
    "brilliant performances and a gripping story from start to finish",
    "one of the best films I have seen this year, truly excellent",
    "the acting was superb and the plot kept me hooked throughout",
    "a wonderful, heartwarming movie that I would happily watch again",
    "amazing cinematography and a genuinely touching story",
    "the cast delivered outstanding performances, highly recommended",
    "a masterpiece with brilliant direction and an emotional payoff",
    "great movie, funny, smart, and beautifully shot",
    "I really enjoyed this film, the writing was clever and sharp",
]
hand_written_negative = [
    "this movie was boring and a complete waste of time",
    "terrible acting and a plot that made no sense at all",
    "one of the worst films I have ever sat through, awful",
    "the story was dull and the pacing dragged on forever",
    "a disappointing movie with flat, lifeless performances",
    "poor writing and an ending that felt completely unearned",
    "the cast seemed disengaged, the whole thing felt lazy",
    "a mess of a film with no coherent plot or direction",
    "bad movie, unfunny, predictable, and badly shot",
    "I did not enjoy this film, the dialogue was clunky and forced",
]

positive_adjectives = ["fantastic", "brilliant", "excellent", "superb", "wonderful", "amazing",
                        "outstanding", "incredible", "delightful", "captivating", "charming",
                        "masterful", "exceptional", "gripping", "touching"]
negative_adjectives = ["terrible", "boring", "awful", "dull", "disappointing", "lifeless",
                        "poor", "clunky", "confusing", "forgettable", "tedious", "weak",
                        "flat", "hollow", "depressing"]
templates = [
    "this movie was {adj} and I {feel} every minute of it",
    "the acting was {adj} and the story was {adj2}",
    "a {adj} film with {adj2} performances from the whole cast",
    "the plot was {adj}, and the pacing felt {adj2}",
    "I found the movie to be {adj}, with {adj2} writing throughout",
    "the director delivered a {adj} experience with {adj2} visuals",
    "overall this was a {adj} film, {adj2} from start to finish",
    "the soundtrack was {adj} and the ending felt {adj2}",
    "critics called it {adj}, and I found it genuinely {adj2}",
    "the characters were {adj}, making for a {adj2} watch",
]

def generate_reviews(adjectives, feel_word, n):
    reviews = []
    for _ in range(n):
        template = random.choice(templates)
        reviews.append(template.format(
            adj=random.choice(adjectives), adj2=random.choice(adjectives), feel=feel_word,
        ))
    return reviews

generated_positive = generate_reviews(positive_adjectives, "loved", 60)
generated_negative = generate_reviews(negative_adjectives, "hated", 60)

texts = hand_written_positive + generated_positive + hand_written_negative + generated_negative
labels = ([1] * (len(hand_written_positive) + len(generated_positive)) +
          [0] * (len(hand_written_negative) + len(generated_negative)))

df = pd.DataFrame({"text": texts, "label": labels}).drop_duplicates(subset="text").reset_index(drop=True)

noise_rng = np.random.RandomState(1)
n_noisy = int(0.06 * len(df))
noisy_indices = noise_rng.choice(df.index, size=n_noisy, replace=False)
df.loc[noisy_indices, "label"] = 1 - df.loc[noisy_indices, "label"]

print(f"Total examples: {len(df)}  |  Positive: {df['label'].sum()}  |  Negative: {(df['label']==0).sum()}")


In [ ]:
# ── Building a vocabulary and turning text into integer ID sequences ────────
# Neural networks need NUMBERS, not strings. Instead of TF-IDF's one giant
# sparse vector per document, sequence models want one INTEGER PER TOKEN,
# in order — the network itself learns a dense vector for each integer
# (via an Embedding layer, Part 4) rather than us hand-crafting features.

from collections import Counter

def tokenize(text):
    return text.lower().split()   # a naive whitespace tokenizer, good enough
                                   # for this teaching example; swap in a real
                                   # tokenizer (Module 2 / a subword tokenizer,
                                   # Module 7) for production use

word_counts = Counter()
for text in df["text"]:
    word_counts.update(tokenize(text))

# Reserve index 0 for padding and 1 for unknown/out-of-vocabulary words —
# both are standard conventions you'll see in almost every NLP codebase.
PAD_IDX, UNK_IDX = 0, 1
vocab = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}
for word, count in word_counts.most_common():
    if count >= 1:                # min_count threshold, same idea as Module 4's Word2Vec
        vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")

def encode(text, vocab, max_len=20):
    """Turn a sentence into a fixed-length list of integer token IDs."""
    tokens = tokenize(text)[:max_len]                                 # truncate long sequences
    ids = [vocab.get(tok, UNK_IDX) for tok in tokens]                 # unknown words -> UNK_IDX
    ids += [PAD_IDX] * (max_len - len(ids))                            # pad short sequences with 0s
    return ids

MAX_LEN = 20
example_encoded = encode(df["text"].iloc[0], vocab, MAX_LEN)
print("Example sentence:", df['text'].iloc[0])
print("Encoded as IDs:  ", example_encoded)


In [ ]:
# ── Train/test split, then wrap in a PyTorch Dataset + DataLoader ───────────
from sklearn.model_selection import train_test_split

train_texts, test_texts, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.25, random_state=RANDOM_STATE, stratify=df["label"]
)

class SentimentDataset(Dataset):
    """
    📋 COPY-PASTE TEMPLATE
    A standard PyTorch Dataset: __len__ tells PyTorch how many examples
    exist, __getitem__ returns ONE (input, label) pair. DataLoader (below)
    handles batching, shuffling, and iteration on top of this.
    """
    def __init__(self, texts, labels, vocab, max_len):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = encode(self.texts[idx], self.vocab, self.max_len)
        return torch.tensor(encoded, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float32)

train_dataset = SentimentDataset(train_texts, y_train, vocab, MAX_LEN)
test_dataset = SentimentDataset(test_texts, y_test, vocab, MAX_LEN)

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)   # shuffle TRAIN only
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)     # never shuffle test/eval

batch_inputs, batch_labels = next(iter(train_loader))
print("One batch of inputs shape:", batch_inputs.shape)   # (batch_size, MAX_LEN)
print("One batch of labels shape:", batch_labels.shape)   # (batch_size,)


## Part 3 — RNN Fundamentals

An RNN processes a sequence **one token at a time**. At each step, it combines the current token's vector with a **hidden state** carried over from the previous step, producing a new hidden state:

$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$$

- $x_t$ = the current token's embedding vector
- $h_{t-1}$ = the hidden state summarizing everything read *before* this token
- $h_t$ = the updated hidden state, now also summarizing the current token

The **same weights** ($W_x$, $W_h$) are reused at every single time step — this weight-sharing is what lets an RNN handle sentences of *any* length with a *fixed* number of parameters. After the last token, the final hidden state $h_T$ is treated as a summary of the whole sentence, ready to feed into a classifier head.

### The catch: vanishing gradients
During training, the same weight matrix gets multiplied together over and over across every time step when computing gradients (backpropagation through time). Over long sequences, gradients can shrink toward zero (or explode) exponentially — meaning a plain RNN effectively "forgets" anything from many steps back by the time it reaches the end of a long sentence. Part 5 (LSTM/GRU) exists specifically to fix this.


## Part 4 — Building & Training an RNN Text Classifier

Three pieces, stacked: an **Embedding layer** (turns each integer token ID into a dense learned vector — this is conceptually the same idea as Module 4's Word2Vec, except here the vectors are learned *jointly* with the classifier, end-to-end, rather than pretrained separately), an **RNN layer** (reads the sequence of embeddings, produces a final hidden state), and a **Linear classifier head** (turns that hidden state into a prediction).


In [ ]:
class RNNClassifier(nn.Module):
    """
    📋 COPY-PASTE TEMPLATE — swap the nn.RNN line for nn.LSTM or nn.GRU
    (Part 5) and everything else in this class stays identical, which is
    exactly why those swaps are so easy in practice.
    """
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        # padding_idx tells PyTorch: never update this row's embedding, and
        # treat it as a fixed zero vector — pure padding shouldn't influence learning

        self.rnn = nn.RNN(
            input_size=embedding_dim,    # size of each token's embedding vector
            hidden_size=hidden_dim,      # size of the hidden state carried across time steps
            batch_first=True,             # our tensors are shaped (batch, sequence, features);
                                           # without this, PyTorch defaults to (sequence, batch, features)
        )
        self.classifier = nn.Linear(hidden_dim, 1)   # 1 output = a single logit for binary classification

    def forward(self, token_ids):
        embedded = self.embedding(token_ids)              # (batch, seq_len) -> (batch, seq_len, embedding_dim)
        rnn_output, final_hidden = self.rnn(embedded)       # final_hidden: (1, batch, hidden_dim) —
                                                              # the summary of the WHOLE sequence
        summary = final_hidden.squeeze(0)                    # -> (batch, hidden_dim)
        logits = self.classifier(summary)                     # -> (batch, 1) raw, unnormalized score
        return logits.squeeze(1)                               # -> (batch,) — one logit per example

rnn_model = RNNClassifier(vocab_size=len(vocab))
print(rnn_model)


In [ ]:
# ── The training loop (this exact shape works for every model in this notebook) ─

def train_model(model, train_loader, test_loader, epochs=15, lr=0.01):
    """
    📋 COPY-PASTE TEMPLATE — the standard PyTorch training loop. This
    function's BODY never really changes between projects; only the model
    architecture passed in does.
    """
    criterion = nn.BCEWithLogitsLoss()   # combines a sigmoid + binary cross-entropy in one
                                          # numerically stable step — standard for binary
                                          # classification when your model outputs raw logits
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "test_accuracy": []}

    for epoch in range(epochs):
        model.train()                     # switch to TRAINING mode (matters for dropout/batchnorm)
        total_loss = 0.0
        for batch_inputs, batch_labels in train_loader:
            optimizer.zero_grad()          # clear gradients from the previous step
            logits = model(batch_inputs)
            loss = criterion(logits, batch_labels)
            loss.backward()                 # compute gradients via backpropagation
            optimizer.step()                 # update weights using those gradients
            total_loss += loss.item()

        # ── Evaluate on the test set after each epoch ────────────────────────
        model.eval()                        # switch to EVAL mode
        correct, total = 0, 0
        with torch.no_grad():                # no gradient tracking needed during evaluation —
                                               # saves memory and computation
            for batch_inputs, batch_labels in test_loader:
                logits = model(batch_inputs)
                predictions = (torch.sigmoid(logits) > 0.5).float()   # sigmoid turns
                                                                        # logits into
                                                                        # probabilities
                correct += (predictions == batch_labels).sum().item()
                total += batch_labels.size(0)

        avg_loss = total_loss / len(train_loader)
        accuracy = correct / total
        history["train_loss"].append(avg_loss)
        history["test_accuracy"].append(accuracy)
        print(f"Epoch {epoch+1:2d}/{epochs}  |  train loss: {avg_loss:.4f}  |  test accuracy: {accuracy:.3f}")

    return history

rnn_history = train_model(rnn_model, train_loader, test_loader, epochs=15)


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(rnn_history["train_loss"])
plt.title("Training Loss")
plt.xlabel("Epoch")

plt.subplot(1, 2, 2)
plt.plot(rnn_history["test_accuracy"])
plt.title("Test Accuracy")
plt.xlabel("Epoch")
plt.tight_layout()
plt.show()

# Reading this: loss should trend down, accuracy should trend up. If accuracy
# plateaus early or bounces around noisily, that's usually a sign the dataset
# is too small (as ours deliberately is, for teaching speed) or the learning
# rate needs adjusting.


## Part 5 — LSTM & GRU: Fixing the Vanishing Gradient

**LSTM (Long Short-Term Memory)** adds a separate **cell state** (a kind of conveyor belt running through the whole sequence) plus three **gates** that learn, at every step, what to keep and what to throw away:
- **Forget gate**: how much of the old cell state to discard
- **Input gate**: how much of the new information to add in
- **Output gate**: how much of the cell state to expose as the current hidden state

Because the cell state update is mostly additive (rather than repeated matrix multiplication like a plain RNN), gradients can flow much further back through time without vanishing — this is THE reason LSTMs could handle meaningfully longer sequences than plain RNNs, and why they dominated NLP for roughly a decade before Transformers.

**GRU (Gated Recurrent Unit)** simplifies this to two gates (reset + update) and no separate cell state — fewer parameters, faster to train, and empirically competitive with LSTM on many tasks. Common rule of thumb: try GRU first for speed; reach for LSTM if you need the extra modeling capacity and have the compute budget.


In [ ]:
class LSTMClassifier(nn.Module):
    """
    📋 COPY-PASTE TEMPLATE — identical structure to RNNClassifier above;
    ONLY the middle layer changed from nn.RNN to nn.LSTM. This drop-in
    swap is exactly how easy it is in a real project.
    """
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, token_ids):
        embedded = self.embedding(token_ids)
        # LSTM returns (output_per_step, (final_hidden_state, final_cell_state)) —
        # note the EXTRA cell state compared to plain RNN's single hidden state
        lstm_output, (final_hidden, final_cell) = self.lstm(embedded)
        summary = final_hidden.squeeze(0)
        logits = self.classifier(summary)
        return logits.squeeze(1)

lstm_model = LSTMClassifier(vocab_size=len(vocab))
lstm_history = train_model(lstm_model, train_loader, test_loader, epochs=15)


In [ ]:
# 📋 COPY-PASTE TEMPLATE — GRU version (swap nn.LSTM for nn.GRU, note GRU
# has NO separate cell state, so forward() returns just one hidden state)
#
# class GRUClassifier(nn.Module):
#     def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64, pad_idx=0):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
#         self.gru = nn.GRU(input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True)
#         self.classifier = nn.Linear(hidden_dim, 1)
#
#     def forward(self, token_ids):
#         embedded = self.embedding(token_ids)
#         gru_output, final_hidden = self.gru(embedded)   # only ONE hidden state, like plain RNN
#         summary = final_hidden.squeeze(0)
#         logits = self.classifier(summary)
#         return logits.squeeze(1)

print("GRU template shown above — the middle ground between RNN's simplicity and LSTM's capacity.")


## Part 6 — Bidirectional RNNs

Everything so far reads left-to-right only, so at token 3 the model has no idea what token 10 says yet. For **classification** (where you see the whole sentence before deciding), that's an unnecessary restriction — a **bidirectional** RNN/LSTM runs one pass left-to-right and a second pass right-to-left, then concatenates both final hidden states, so every position's representation is informed by context from *both* directions.

⚠️ Bidirectional models only make sense when the **full sequence is available upfront** — they're wrong for tasks like real-time text generation or live speech transcription, where future tokens genuinely don't exist yet at prediction time.


In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,   # the ONE new argument — doubles the output dimension,
                                  # since forward and backward hidden states get concatenated
        )
        self.classifier = nn.Linear(hidden_dim * 2, 1)   # * 2 because bidirectional concatenates
                                                          # the forward AND backward final states

    def forward(self, token_ids):
        embedded = self.embedding(token_ids)
        lstm_output, (final_hidden, final_cell) = self.lstm(embedded)
        # final_hidden shape: (2, batch, hidden_dim) — index 0 is the forward
        # direction's final state, index 1 is the backward direction's
        forward_state = final_hidden[0]
        backward_state = final_hidden[1]
        summary = torch.cat([forward_state, backward_state], dim=1)   # -> (batch, hidden_dim * 2)
        logits = self.classifier(summary)
        return logits.squeeze(1)

bilstm_model = BiLSTMClassifier(vocab_size=len(vocab))
bilstm_history = train_model(bilstm_model, train_loader, test_loader, epochs=15)


## Part 7 — Sequence Labeling: Tagging Every Token

So far every model produces **one** prediction per sentence (sentiment). The other major RNN use case is **sequence labeling**: producing **one prediction per token** — Named Entity Recognition (is each word a PERSON/ORG/LOCATION/none?), Part-of-Speech tagging (is each word a NOUN/VERB/ADJ?). The architecture change is small: instead of taking only the FINAL hidden state, you take the hidden state **at every time step** and classify each one independently.


In [ ]:
class SequenceLabeler(nn.Module):
    """
    📋 COPY-PASTE TEMPLATE for NER/POS-tagging-style tasks.
    The only structural difference from the classifiers above: the Linear
    layer is applied to EVERY time step's output, not just the last one.
    """
    def __init__(self, vocab_size, num_tags, embedding_dim=32, hidden_dim=64, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embedding_dim, hidden_size=hidden_dim,
            batch_first=True, bidirectional=True,   # bidirectional is especially
                                                      # valuable for tagging — knowing
                                                      # the word AFTER "New" helps decide
                                                      # if "York" makes it a location
        )
        self.tag_classifier = nn.Linear(hidden_dim * 2, num_tags)

    def forward(self, token_ids):
        embedded = self.embedding(token_ids)
        lstm_output, _ = self.lstm(embedded)          # KEEP the per-step output this time,
                                                        # (batch, seq_len, hidden_dim * 2) —
                                                        # NOT just the final hidden state
        tag_logits = self.tag_classifier(lstm_output)   # -> (batch, seq_len, num_tags) —
                                                          # one prediction PER TOKEN
        return tag_logits

# Example: 5 possible tags (e.g. O, PERSON, ORG, LOCATION, MISC — the standard
# NER tag set shape), just to show the model builds and runs correctly:
tagger = SequenceLabeler(vocab_size=len(vocab), num_tags=5)
sample_batch, _ = next(iter(train_loader))
sample_output = tagger(sample_batch)
print("Input shape: ", sample_batch.shape)          # (batch, seq_len)
print("Output shape:", sample_output.shape)          # (batch, seq_len, num_tags)
# In a real NER project, the loss function is CrossEntropyLoss applied to
# EVERY token position at once (with padding positions masked out of the loss).


## Part 8 — Attention: the Idea That Broke the Bottleneck

Even a bidirectional LSTM still squeezes an entire sentence through **one fixed-size hidden vector** by the end — a real bottleneck for long documents (imagine summarizing an entire paragraph into 64 numbers and hoping nothing important got lost). **Attention** fixes this by letting the model look back at **every** token's hidden state when making each decision, weighted by how *relevant* each one is right now — instead of relying on a single final summary.

$$\text{score}(h_t) = h_t \cdot q \qquad \alpha_t = \text{softmax}(\text{score}(h_t)) \qquad \text{context} = \sum_t \alpha_t h_t$$

- $q$ = a "query" vector representing what we're currently trying to decide
- $\alpha_t$ = how much attention to pay to time step $t$ (all $\alpha_t$ sum to 1, via softmax)
- The final **context vector** is a weighted average of ALL hidden states, not just the last one

This is a genuinely pivotal idea: once researchers noticed attention alone (without any recurrence at all) could model relationships between tokens, that observation led directly to the 2017 paper *"Attention Is All You Need"* — the Transformer, and the architecture underlying every modern LLM.


In [ ]:
class AttentionClassifier(nn.Module):
    """
    📋 COPY-PASTE TEMPLATE — a minimal, from-scratch attention layer bolted
    onto a BiLSTM, so you can see exactly how the weighted-average mechanism
    above turns into real code before Module 7 shows you the full Transformer.
    """
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention_weights = nn.Linear(hidden_dim * 2, 1)   # learns a single
                                                                  # relevance score per time step
        self.classifier = nn.Linear(hidden_dim * 2, 1)

    def forward(self, token_ids):
        embedded = self.embedding(token_ids)
        lstm_output, _ = self.lstm(embedded)             # (batch, seq_len, hidden_dim * 2) —
                                                            # every time step's hidden state, kept

        scores = self.attention_weights(lstm_output)        # (batch, seq_len, 1) — one relevance
                                                              # score per token
        attention_alphas = torch.softmax(scores, dim=1)       # normalize scores across the
                                                                # SEQUENCE dimension so they sum to 1

        # Weighted sum: multiply each time step's hidden state by its
        # attention weight, then sum across the sequence dimension —
        # this IS the "context vector" from the formula above.
        context = torch.sum(attention_alphas * lstm_output, dim=1)   # -> (batch, hidden_dim * 2)

        logits = self.classifier(context)
        return logits.squeeze(1), attention_alphas.squeeze(-1)   # return the weights too,
                                                                    # so we can visualize them

attention_model = AttentionClassifier(vocab_size=len(vocab))

# This model's forward() returns a tuple, so it needs a slightly different
# training loop — a one-line change from train_model() above:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(attention_model.parameters(), lr=0.01)
for epoch in range(15):
    attention_model.train()
    for batch_inputs, batch_labels in train_loader:
        optimizer.zero_grad()
        logits, _ = attention_model(batch_inputs)   # unpack the tuple; ignore attention weights during training
        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()

attention_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for batch_inputs, batch_labels in test_loader:
        logits, _ = attention_model(batch_inputs)
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == batch_labels).sum().item()
        total += batch_labels.size(0)
print(f"Attention model test accuracy: {correct/total:.3f}")


In [ ]:
# ── Visualizing what the model paid attention to ─────────────────────────────
# This is the payoff of attention over a plain RNN: you can literally SEE
# which words the model leaned on for its decision — genuine interpretability
# that a plain LSTM's single final hidden vector never gave you.

sample_text = test_texts.iloc[0]
sample_encoded = torch.tensor([encode(sample_text, vocab, MAX_LEN)], dtype=torch.long)

attention_model.eval()
with torch.no_grad():
    logits, alphas = attention_model(sample_encoded)

tokens = tokenize(sample_text)[:MAX_LEN]
weights = alphas[0][:len(tokens)].numpy()

plt.figure(figsize=(10, 2))
plt.bar(range(len(tokens)), weights)
plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right")
plt.title(f"Attention weights — predicted: {'positive' if torch.sigmoid(logits).item() > 0.5 else 'negative'}")
plt.tight_layout()
plt.show()


## Part 9 — Why Transformers Replaced RNNs

Attention (Part 8) solved the "everything squeezed through one vector" problem. But one core limitation of RNNs/LSTMs remained completely untouched: **you cannot compute step 5's hidden state until you've computed step 4's** — recurrence is inherently **sequential**. On modern GPUs/TPUs built for massive parallel computation, that sequential dependency is a huge waste of available hardware, and it makes training on internet-scale text painfully slow.

The 2017 insight behind Transformers: **if attention alone can capture relationships between tokens, drop the recurrence entirely** — let the model look at every token in the sequence *simultaneously*, computing all positions' attention in parallel. That single architectural change is what unlocked training on the scale of data and compute that today's LLMs (including Claude) are built on.

| | RNN/LSTM | Transformer |
|---|-----------|---------------|
| Processes tokens | One at a time, in order | All at once, in parallel |
| Long-range dependencies | Degrade with distance (even with LSTM gating) | Direct connection between any two tokens, regardless of distance |
| Training speed at scale | Slow — sequential bottleneck | Fast — fully parallelizable on GPUs/TPUs |
| Still used today for... | Streaming/online inference, resource-constrained edge devices, time-series-like sequential data | Essentially everything at scale — the default for text |

### A realistic comparison, on OUR dataset


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# The Module 5 classical baseline, side-by-side with what we just built:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english", min_df=2)
X_train_tfidf = vectorizer.fit_transform(train_texts)
X_test_tfidf = vectorizer.transform(test_texts)

logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_tfidf, y_train)
logreg_accuracy = (logreg.predict(X_test_tfidf) == y_test.values).mean()

print(f"Logistic Regression (Module 5):  {logreg_accuracy:.3f}")
print(f"Plain RNN:                        {rnn_history['test_accuracy'][-1]:.3f}")
print(f"LSTM:                             {lstm_history['test_accuracy'][-1]:.3f}")
print(f"Bidirectional LSTM:                {bilstm_history['test_accuracy'][-1]:.3f}")

# A important, honest lesson: on a TINY dataset like ours, classical
# TF-IDF + Logistic Regression very often matches or beats these neural
# models, because neural networks need substantially more data to learn
# good representations from scratch. RNNs/LSTMs/Transformers pull decisively
# ahead once you have thousands-to-millions of training examples -- exactly
# the regime that made "Attention Is All You Need" (2017) transformative.


## Part 10 — Production Patterns for Sequence Models

Three engineering details every production PyTorch NLP codebase handles, that this teaching notebook simplified for clarity above.


In [ ]:
# ── Padding efficiently with pack_padded_sequence ────────────────────────────
# Above, we padded every sequence to a FIXED length (20) and just let the RNN
# process the padding tokens too — wasteful at scale (computing over pure
# padding for no benefit). Production code instead tells PyTorch the REAL
# length of each sequence, so it skips computation on the padded positions.

# from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
#
# def forward(self, token_ids, lengths):
#     embedded = self.embedding(token_ids)
#     packed = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
#     packed_output, (final_hidden, final_cell) = self.lstm(packed)
#     output, _ = pad_packed_sequence(packed_output, batch_first=True)   # unpack if you
#                                                                        # need per-step outputs
#     summary = final_hidden.squeeze(0)   # final_hidden is already correct --
#                                          # pack_padded_sequence ensures it reflects
#                                          # each sequence's TRUE last token, not padding
#     return self.classifier(summary)

print("pack_padded_sequence pattern shown above -- the standard production efficiency fix.")


In [ ]:
# ── Gradient clipping (prevents exploding gradients during training) ────────
# Long sequences can occasionally produce huge gradient values that destabilize
# training. Clipping caps the gradient norm before each optimizer step.

# torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
# # ... called right after loss.backward() and right before optimizer.step()

print("Gradient clipping pattern shown above -- add this if you see loss spike to NaN mid-training.")


In [ ]:
# ── Saving and loading a trained PyTorch model checkpoint ───────────────────

# Save just the learned weights (the standard, recommended approach):
# torch.save(lstm_model.state_dict(), "lstm_sentiment_model.pt")

# Load into a freshly-constructed model with the SAME architecture:
# loaded_model = LSTMClassifier(vocab_size=len(vocab))
# loaded_model.load_state_dict(torch.load("lstm_sentiment_model.pt"))
# loaded_model.eval()   # always call .eval() before running inference --
#                        # disables dropout/batchnorm training behavior

# 📋 COPY-PASTE TEMPLATE -- a full checkpoint saving BOTH weights and the
# vocabulary/config needed to reconstruct the model elsewhere:
# checkpoint = {
#     "model_state_dict": lstm_model.state_dict(),
#     "vocab": vocab,
#     "max_len": MAX_LEN,
#     "embedding_dim": 32,
#     "hidden_dim": 64,
# }
# torch.save(checkpoint, "sentiment_model_checkpoint.pt")
#
# loaded = torch.load("sentiment_model_checkpoint.pt")
# model = LSTMClassifier(vocab_size=len(loaded["vocab"]))
# model.load_state_dict(loaded["model_state_dict"])

print("Checkpoint save/load patterns shown above -- always save the vocab alongside the "
      "weights, or a reloaded model has no idea how to turn text back into token IDs.")


### 🔀 When to still reach for an RNN/LSTM today, instead of a Transformer
| Situation | Why RNN/LSTM can still be the right call |
|-----------|---------------------------------------------|
| True streaming/online inference (must predict before the full sequence exists) | RNNs process token-by-token naturally; Transformers need the sequence available (or careful causal-masking + KV-caching engineering) |
| Extremely resource-constrained deployment (tiny edge devices) | Smaller, cheaper, lower-latency per step for short sequences |
| Genuinely small labeled datasets | As Part 9 showed — fewer parameters can mean less overfitting on tiny data |
| Long numeric/sensor time-series (not text) | RNNs remain a very reasonable default outside NLP specifically |

For anything text-related at real production scale today, the honest default is a Transformer (Module 7) — often a pretrained one you fine-tune rather than train from scratch.


## Recap & What's Next

You built and trained an RNN, an LSTM, a bidirectional LSTM, a sequence labeler, and an attention-based classifier — all in PyTorch, all from the ground up, with a training loop template you'll reuse for the rest of this course. You also got an honest, data-backed answer to "why bother with all this if classical ML can be just as good on small data?" — it's the *scale* regime where these architectures pull ahead, which is exactly the regime Transformers were built to exploit even further.

### Try this before the next lesson
1. Swap `nn.RNN` for `nn.GRU` in the template from Part 5 and compare test accuracy against the LSTM.
2. Increase `hidden_dim` and `embedding_dim` and see whether it helps or just overfits faster on our small dataset.
3. Pick one misclassified test example and look at its attention weights (Part 8) — does the visualization explain the mistake?

### Next lesson in your NLP mastery path
**Module 7: The Transformer Architecture, End to End** — self-attention and multi-head attention from scratch, positional encoding (since dropping recurrence means the model needs another way to know word order), the encoder/decoder structure, and exactly how this architecture scales up into models like BERT, GPT, and Claude.
